In [1]:
# Global Import and Directories
import sys
sys.path.append("..")
import Setup
from Setup import *

Current working directory: /home/qusta100/STGNN/02_Code/06_Performance_Comparison


In [2]:
# Load main data set
df_thuringia = pd.read_csv(
    Setup.DATA_SCRATCH / "02_Processed_Data" / "02_Analysis" / "Thuringia.csv",
    dtype={9: str}
)
df_thuringia["date"] = pd.to_datetime(df_thuringia["date"])

/var/tmp/pbs.17090655.hpc-batch/ipykernel_137542/3416999999.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_thuringia = pd.read_csv(


In [8]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ----------------------------
# Serie pro Station (15min)
# ----------------------------
def build_series(df_station, date_col="date", y_col="price", freq="15min"):
    s = df_station[[date_col, y_col]].dropna().copy()
    s[date_col] = pd.to_datetime(s[date_col])
    s = s.sort_values(date_col).set_index(date_col)[y_col]

    # erzwingt regelmäßige Frequenz und macht Lücken sichtbar
    s = s.asfreq(freq)

    # SARIMAX braucht keine NaNs
    s = s.ffill().bfill()
    return s


# ----------------------------
# Kleine Order-Suche (optional)
# ----------------------------
def select_sarima_order(y_train, seasonal_period=96, max_evals=120):
    # bewusst klein, sonst wird es teuer
    p_list = [0, 1, 2]
    d_list = [0, 1]
    q_list = [0, 1, 2]
    P_list = [0, 1]
    D_list = [0, 1]
    Q_list = [0, 1]

    best = None
    evals = 0

    for p in p_list:
        for d in d_list:
            for q in q_list:
                for P in P_list:
                    for D in D_list:
                        for Q in Q_list:
                            evals += 1
                            if evals > max_evals:
                                return best

                            order = (p, d, q)
                            seasonal_order = (P, D, Q, seasonal_period)

                            try:
                                m = SARIMAX(
                                    y_train,
                                    order=order,
                                    seasonal_order=seasonal_order,
                                    trend="n",
                                    enforce_stationarity=False,
                                    enforce_invertibility=False,
                                )
                                r = m.fit(disp=False)
                                aic = r.aic

                                if best is None or aic < best["aic"]:
                                    best = {"order": order, "seasonal_order": seasonal_order, "aic": aic}
                            except Exception:
                                continue

    return best


# ----------------------------
# Forecast für Horizonte {1,2,4}
# ----------------------------
def forecast_h1234(res, steps=4):
    # predicted_mean: t+1..t+steps
    pred = res.get_forecast(steps=steps).predicted_mean
    return {
        1: float(pred.iloc[0]),
        2: float(pred.iloc[1]),
        4: float(pred.iloc[3]),
    }


In [9]:
import numpy as np
import pandas as pd
from statsmodels.tsa.statespace.sarimax import SARIMAX

def _fit_one_station(
    sid,
    df_s,
    station_col="station_id",
    date_col="date",
    y_col="price",
    train_end="2019-12-31 23:45:00",
    freq="15min",
    seasonal_period=96,
    grid=True,
    min_train_points=None,
):
    train_end = pd.Timestamp(train_end)
    if min_train_points is None:
        min_train_points = seasonal_period * 3

    s = build_series(df_s, date_col=date_col, y_col=y_col, freq=freq)

    if (s.index.min() > train_end) or (s.index.max() <= train_end + pd.Timedelta(minutes=60)):
        return None

    y_train = s.loc[:train_end]
    y_after = s.loc[train_end:].iloc[1:]  # direkt nach train_end

    try:
        y_true = {1: float(y_after.iloc[0]), 2: float(y_after.iloc[1]), 4: float(y_after.iloc[3])}
    except Exception:
        return None

    # Fallback wenn zu kurz
    if len(y_train) < min_train_points:
        base = y_train
        pred = {}
        for h in [1, 2, 4]:
            idx = -seasonal_period + (h - 1)
            if abs(idx) <= len(base):
                pred[h] = float(base.iloc[idx])
            else:
                pred[h] = float(base.iloc[-1])

        return {
            "station_id": sid,
            "mae_h1": abs(y_true[1] - pred[1]),
            "mae_h2": abs(y_true[2] - pred[2]),
            "mae_h4": abs(y_true[4] - pred[4]),
            "order": None,
            "seasonal_order": None,
            "aic": None,
            "note": "fallback_seasonal_naive_short_series",
        }

    # Order wählen
    if grid:
        best = select_sarima_order(y_train, seasonal_period=seasonal_period)
        if best is None:
            order, seasonal_order, aic = (1, 1, 1), (1, 1, 1, seasonal_period), None
            note = "grid_failed_used_default"
        else:
            order, seasonal_order, aic = best["order"], best["seasonal_order"], best["aic"]
            note = None
    else:
        order, seasonal_order, aic, note = (1, 1, 1), (1, 1, 1, seasonal_period), None, None

    try:
        model = SARIMAX(
            y_train,
            order=order,
            seasonal_order=seasonal_order,
            trend="n",
            enforce_stationarity=False,
            enforce_invertibility=False,
        )
        res = model.fit(disp=False)
        pred = forecast_h1234(res, steps=4)

        return {
            "station_id": sid,
            "mae_h1": abs(y_true[1] - pred[1]),
            "mae_h2": abs(y_true[2] - pred[2]),
            "mae_h4": abs(y_true[4] - pred[4]),
            "order": order,
            "seasonal_order": seasonal_order,
            "aic": aic,
            "note": note,
        }
    except Exception as e:
        return {
            "station_id": sid,
            "mae_h1": np.nan,
            "mae_h2": np.nan,
            "mae_h4": np.nan,
            "order": order,
            "seasonal_order": seasonal_order,
            "aic": aic,
            "note": f"fit_error:{type(e).__name__}",
        }


In [10]:
def run_sarima_per_station_parallel(
    df,
    station_col="station_id",
    date_col="date",
    y_col="price",
    train_end="2019-12-31 23:45:00",
    freq="15min",
    seasonal_period=96,
    grid=True,
    min_train_points=None,
    n_jobs=-1,
    batch_size="auto",
    prefer="processes",
):
    grouped = list(df.groupby(station_col, sort=False))

    results = Parallel(
        n_jobs=n_jobs,
        batch_size=batch_size,
        prefer=prefer,
        backend="loky",  # explizit Prozesse
    )(
        delayed(_fit_one_station)(
            sid,
            df_s,
            station_col=station_col,
            date_col=date_col,
            y_col=y_col,
            train_end=train_end,
            freq=freq,
            seasonal_period=seasonal_period,
            grid=grid,
            min_train_points=min_train_points,
        )
        for sid, df_s in grouped
    )

    rows = [r for r in results if r is not None]
    out = pd.DataFrame(rows)
    if len(out) == 0:
        return out
    out["mae_mean_124"] = out[["mae_h1", "mae_h2", "mae_h4"]].mean(axis=1)
    return out

In [ ]:
out = run_sarima_per_station_parallel(
    df_thuringia,
    station_col="station_uuid",
    date_col="date",
    y_col="e5",
    train_end="2025-04-26 00:00:00",
    seasonal_period=96,
    grid=True,
    n_jobs=12,      
    batch_size=1,
)